# 1.9 Reelle datasett med CSV

**Kompetansemål:**
- bruke programmering til å innhente, analysere og presentere informasjon fra reelle datasett.

## Åpne data
Åpne data er datasett som vi kan bruke gratis, fordi de ikke er begrenset av opphavsrett eller lisenser. Offentlige og andre virksomheter tilbyr åpne data. Det er ingen selvfølge at data er gratis, for det ligger mye penger i dem, se *The world's most valuable resource is no longer oil, but data* (The Economist). På nettet finnes mye åpne data som vi kan bruke, og [https://data.norge.no/](https://data.norge.no/) (Felles datakatalog) er et greit utgangspunkt. Vi kan laste ned filer eller la programmet vårt hente informasjon via API *(Application Programming Interface)*.

I denne bolken skal vi bruke **Statens satser utland 2019** som viser statens satser for dekning av utgifter til reise og kost utenfor Norge. La oss si vi ønsker å finne de fem dyreste og de fem billigste stedene, samt gi en grafisk presentasjon av dataene.

Filtypen er **.csv**. Det finnes også informasjon om datasettet. Når vi jobber med store datasett, er det vanlig å foreta en foreløpig undersøkelse av datasettet, for å få et førsteinntrykk og bli bedre kjent med innholdet. Vi kan se på filen (2019.csv) i VS Code. Vi ser at det er én overskriftsrad og at skilletegnet er semikolon (`;`), ikke komma (`,`). Flere rader mangler `by` og `administrativt_satt`, og en rad har ikke `kost`.

```
verdensdel;land;by;kost;administrativt_satt
Afrika;Øvrige områder;;500;
Afrika;Algerie;;500;
...
Amerika;Brasil;;560;
Amerika;Brasil;Rio de Janeiro;620;
...
Europa;Danmark;;1220;
Europa;Danmark;København;1220;
Europa;Færøyene;;1220;Danmark
Europa;Grønland;;1220;Danmark
...
Europa;Øvrige områder;;;
```

Det er vanlig å "vaske" dataene før vi bruker dem. Vi må blant annet bestemme hva vi gjør med rader som har noen verdier, men mangler andre. I vårt tilfelle dropper vi linje 1 (overskriftsraden) og linje 113 (uten kost). Begge linjene vil feile hvis vi prøver å gjøre om **kost** til `int` i en rad når verdien er en tom tekst, `''`, eller inneholder noe annet enn siffer.

## HTTP requests
Vi kan lese inn filen slik vi har lært tidligere, eller vi kan laste den ned i programmet med http requests ([w3schools](https://www.w3schools.com/python/module_requests.asp)).

In [2]:
%pip install requests  -q


Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests  # Husk: pip install requests
url = 'https://raw.githubusercontent.com/datahotellet/dataset-archive/refs/heads/main/datasets/statens-satser/utland/2019/dataset.original.csv'
r = requests.get(url)  # returnerer er et Response objekt


SSLError: HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /datahotellet/dataset-archive/refs/heads/main/datasets/statens-satser/utland/2019/dataset.original.csv (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1081)')))

Dersom noe går galt med forespørselen, får vi feil, for eksempel *ConnectionError*, som vi kan fange opp med unntak. Går alt bra, returner `get`-metoden et `Response`-objekt ([W3Schools](https://www.w3schools.com/python/ref_requests_response.asp)) som vi kan utforske med vår variabel `r`.

In [4]:
print(r)  # HTTP 200 OK success status response code
print(r.content[:60])  # vis første 60 bytes
print(r.text[:74])     # vis først 74 tegn


<Response [200]>
b'verdensdel;land;by;kost;administrativt_satt\r\nAfrika;\xc3\x98vrige '
verdensdel;land;by;kost;administrativt_satt
Afrika;Øvrige områder;;560;



`content`-egenskapen viser innholdet som bytes, og vi kan se `\r\n` mellom linjene og 'Ø' kodet som `\xc3\x98`. `text`-egenskapen viser innholdet som tekst. Dersom `r.encoding` ikke er satt, tipper programmet på beste tegnkoding (utf-8) for oss.

Verdiene i teksten kan vi legge inn i en tabell. Så kan vi utføre en datavask, konvertere tekst til tall og sortere tabellen. Vi begynner med å lage en liste med linjene.

In [5]:
linjer = r.text.split('\r\n')  # lag en liste med linjene
print(linjer[:2])              # vis 2 først elementer i listen


['verdensdel;land;by;kost;administrativt_satt', 'Afrika;Øvrige områder;;560;']


Så lager vi en liste av hver linje og vasker bort rader som ikke har tall (som tekst) i **kost**-feltet samtidig som vi konverterer tekst til tall. Vi får da en todimensjonal tabell (lister i liste).

In [7]:
tabell = []

for linje in linjer:
    liste = linje.split(';')  # lag en liste for hver linje

    # Overskriftsrad og rader uten kost tas ikke med (1 og 113)
    if liste[3].isnumeric():
        liste[3] = int(liste[3])
        tabell.append(liste)

print(tabell[0:2])  # vis to første rader i tabellen


[['Afrika', 'Øvrige områder', '', 560, ''], ['Afrika', 'Algerie', '', 500, '']]


Før vi sorterer tabellen skal vi se nærmere på anonyme funksjoner.

## Anonyme funksjoner
Anonyme funksjoner (Lambda *functions*) er funksjoner uten navn som ofte blir brukt som argumenter når vi kaller andre funksjoner. Husk at en funksjons **parametere** er navn som er oppført i funksjonens definisjon. **Argumentene** er de virkelige verdiene som sendes til funksjonen. Vi har tidligere sett at vi kan sortere lister med tall fra lavest til høyest eller tekst alfabetisk.

In [8]:
farger = ['oransje', 'gul', 'grønn']
farger.sort()
print(farger)


['grønn', 'gul', 'oransje']


Hvis vi ønsker å sortere tabellen etter andre kriterier, kan vi lage en funksjon for dette.

In [10]:
def ordlengde(ord):
    return len(ord)

print(ordlengde('Hvit'))

farger = ['oransje', 'gul', 'grønn']
farger.sort(key = ordlengde)
print(farger)


4
['gul', 'grønn', 'oransje']


I stedet for å lage funksjonen `ordlengde()`, kan vi sende den anonyme (lambda) funksjonen som argument til `sort`-metoden, og vi slipper å skrive like mye.

In [ ]:
farger = ['oransje', 'gul', 'grønn']
farger.sort(reverse=True, key = lambda x :len(x))
print(farger)


['gul', 'grønn', 'oransje']


Formatet er `lambda parametere : uttrykk`. Vi kan nå sortere tabellen for statens satser utland etter verdiene i **kost**-feltet, kolonne 4, indeks 3.

In [12]:
sortert_tabell = sorted(tabell, reverse=True, key=lambda x: x[3])
for i in range(5): print(sortert_tabell[i])


['Amerika', 'Venezuela', '', 1600, '']
['Europa', 'Liechtenstein', '', 1240, '']
['Europa', 'Sveits', 'Geneve', 1240, '']
['Europa', 'Sveits', 'Zürich', 1240, '']
['Europa', 'Danmark', '', 1220, '']


Før vi formaterer utskriften, skal vi se på valgoperatøren.

## Valgoperatøren
Hvis vi ønsker at en variabel skal få én av to mulige verdier, kan vi bruke valgoperatøren *(Condtionals Expressions/ternary operator)*. Nedenfor vises et eksempel som skriver ut antall timer, men hvis antallet er 1, heter det jo time i entall.

In [21]:
for i in range(4):
    tidsenhet = 'timer' if i!=1 else 'time'
    print(f'{i} {tidsenhet}')

print()

for i in range(4):
    print(f'{i} {"timer" if i!=1 else "time"}')


0 timer
1 time
2 timer
3 timer

0 timer
1 time
2 timer
3 timer


I uttrykket `x if B else y` blir først betingelsen B evaluert. Hvis B er sann, blir `x` evaluert og verdien returnert, ellers blir `y` evaluert og verdien returnert.

I tabellen for statens satser utland ønsker vi å skrive `by, land` hvis `by` ikke er en tom tekst, ellers skriver vi bare `land`.

In [22]:
print("\n***    De 5 dyreste stedene   ***")
print("Sted                      Kost")
print("*********************************")
for rad in sortert_tabell[:5]:
    sted = f"{rad[2]}, {rad[1]}" if rad[2] != "" else rad[1]
    print(f"{sted:25}{rad[3]:7}")



***    De 5 dyreste stedene   ***
Sted                      Kost
*********************************
Venezuela                   1600
Liechtenstein               1240
Geneve, Sveits              1240
Zürich, Sveits              1240
Danmark                     1220


Vi venter med de 5 billigste stedene til oppgavene. Vi kunne sluppet unna med mindre programmering om vi hadde brukt csv-modulen (.py-fil).

## csv
`csv`-modulen kan lese og skrive csv-filer med tabelldata, så det gir oss litt hjelp på veien. Videre bruker vi en mer kompakt kode for å bygge opp tabellen, og `itemgetter` for å hente verdier fra **kost**-feltet (kolonnen) når vi sorterer. `[:5]` kalles *slicing* (snitte opp) og henter de fem første elementene. Stjerneoperatøren `*` pakker ut den sorterte listen og `sep='\n'` skiller elementene med linjeskift. Det siste er kun gjort for å få hvert element på en ny linje. Vi kunne godt ha formatert utskriften som ovenfor.

In [14]:
import csv
from operator import itemgetter
csv_reader = csv.reader(requests.get(url).text.splitlines(), delimiter=';')
tabell = [x for x in list(csv_reader) if x[3].isnumeric()]
for rad in tabell: rad[3]=int(rad[3])
print(*sorted(tabell, reverse=True, key=itemgetter(3))[:5], sep='\n')


SSLError: HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /datahotellet/dataset-archive/refs/heads/main/datasets/statens-satser/utland/2019/dataset.original.csv (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1081)')))

## pandas
Enklest er det å bruke pakken `pandas`. En pakke er en samling moduler (.py-filer). pandas er godt egnet til arbeide med todimensjonale tabeller og mye annet. Med pandas kan vi løse hele oppgaven med noen få linjer.

In [17]:
import pandas as pd  # Husk: pip install pandas
df = pd.read_csv(url, sep=';')
print(df.sort_values(by='kost', ascending=False).head(5).fillna(''))


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Missing Authority Key Identifier (_ssl.c:1081)>

`read_csv`-metoden returnerer et `DataFrame`-objekt med rader og kolonner som kan sorteres direkte på kolonnenavn fra overskriftsraden. `.head(5)` vil si at vi ønsker kun de første fem radene, og `fillna('')` bytter ut `NaN` *(not a number)* med en tom tekst. I en `DataFrame` betyr `NaN` at det mangler data.

## seaborn
`seaborn` er et bibliotek (pakke) til grafiske fremstillinger. `seaborn` bygger på `matplotlib`, men er enklere å bruke og er tett integrert med pandas.

Et boksplott er et diagram som viser fordelingen av dataene og gir indikasjoner på symmetri i dem. Diagrammet gir et oversiktlig sammendrag så vi kan raskt og effektivt oppsummere og sammenligne forskjellige sett med data.

In [ ]:
import seaborn as sns             # Husk: pip install seaborn
import matplotlib.pyplot as plt   # Husk: pip install matplotlib
sns.set_style('darkgrid')
sns.boxplot(data=df, x='verdensdel', y='kost') \
    .set(title='Statens satser utland 2019')
plt.show()


Streken i midten av boksene viser medianen. En boks strekker seg fra nedre til øvre kvartil, så halvparten av alle verdiene ligger innenfor boksen. Enden på strekene (værhårene/*whiskers*) viser minimum- og maksimum-verdiene, men ikke lenger unna boksen enn 1.5 ganger kvartilbredden (boksens høyde). Verdier på utsiden av værhårene kalles uteliggere og vises med diamant-symboler. Vi må bestemme oss for hva vil gjøre med uteliggere, for verdiene er så avvikende at de kan være feil. Det blir en vurderingssak å tolke hvorvidt uteliggere bør være med i analysen eller ikke.

## Ta med minst dette
Åpne data er gratis og ikke begrenset av opphavsrett eller lisenser. Vi kan hente data (lese filer) fra Internett med bruk av biblioteket `requests` og funksjonen `get`. Dersom det er en CSV-fil, må vi bygge opp tabellen selv. Da er det enklere å bruke biblioteket `csv` som har funksjonen `reader`. Den returnerer et `reader`-objekt som er en *iterator* vi kan bruke i en `for`-løkke. Hver gjennomgang returnerer en linje som en liste med tekststrenger. Enda lettere er det å bruke `pandas`-biblioteket og `read_csv()` som returnerer innholdet i filen som en `DataFrame`. En `DataFrame` er en tabell med rader og kolonner. Med `pandas` er det enkelt å gruppere, sortere og utføre andre operasjoner på tabellen, noe vi skal lære mer om senere og bruke mye. Vi kan bruke `csv.reader()` direkte på en lokal CSV-fil, men når vi bruker den med en URL, som i `csv.reader(requests.get(<url>).text.splitlines())`, hentes teksten som en enkel tekststreng som vi må splitte selv. `pandas.read_csv()`, derimot, tillater både en lokal filsti eller en URL som argument, og håndterer resten automatisk.

Seaborn er et bibliotek tilpasset pandas for å lage diagrammer. Valgoperatøren er ikke noe annet enn å skrive en `if`-blokk på en mer kompakt måte. Når vi bruker funksjoner som krever en annen funksjon som argument, kan vi bruke en anonym `lambda`-funksjon til å definere denne funksjonen direkte i funksjonskallet.

## Øvingsoppgaver

### 1.9.1
Statistisk sentralbyrå tilbyr mye åpne data. Utforsk www.ssb.no for å gjøre deg litt kjent med websidene. Foruten ferdige tabeller kan vi velge ut data selv og laste dem ned som regneark eller med API-spørringer, se eksempelvis *06913: Endringer i kommuner, fylker og hele landets befolkning (K) 1951 - 2022. Statistikkbanken (ssb.no)*.

### 1.9.2
Statistisk sentralbyrå tilbyr også ferdige datasett i csv-format til de mest brukte tabellene. Utforsk hvilke datasett som er tilgjengelig på *API: Ferdige datasett (ssb.no)*.

### 1.9.3
I denne oppgaven skal du analysere samlivsformer i Norge. Gifter flere eller færre seg nå enn tidligere? Lag et program som laster ned csv-filen fra *Personer 18 år og over, etter samlivsform, i prosent. Hele landet, 2005 - siste år*.

Forklar hvor du finner lenken til csv-filen: https://data.ssb.no/api/v0/dataset/86813.csv

Utforsk `seaborn.lineplot` og lag et linjediagram med år langs x-aksen. De ulike samlivsformene skal vises i prosent langs y-akse. Diagrammet skal ha tre linjer; en for hver av samlivsformene.

**TIPS**

Denne filen bruker ikke `utf-8`, men `windows-1252`, og vi bør fortelle om komma som desimaltegn, så slipper vi å gjøre det om senere:
```python
df = pd.read_csv(url, sep=';', decimal=',', encoding='windows-1252')
```

Vi får tak i en kolonne gjennom kolonnenavnet som hentes fra overskriftsraden. Hvis denne teksten er svært lang, som i dette datasettet, kan vi endre på kolonnenavnet eller angi kolonnen ved indeks.
```python
df.iloc[:, 4]  # det samme som
df.columns.values[4] = "prosent"
df['prosent']
```

Hvis årstallene blir gjort om til desimaltall i grafen, er det flere måter å løse det på. Et alternativ er å gjøre årene, som er heltall, om til `Timestamp` objekter.
```python
df["år"] = pd.to_datetime(df["år"].astype(str), format="%Y")
```

In [ ]:
# Skriv koden din her
